# Quantum Energy Levels from Wave Dynamics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gpartin/LFMPublicExperiments/blob/main/LFMPublicExperiments/notebooks/LFM_Quantum_Energy_Levels.ipynb)

## What This Notebook Demonstrates

When a wave is confined in a $\chi$-well (analogous to a quantum box), GOV-01 produces **quantized frequency modes** matching the analytical dispersion relation:

$$\omega_n^2 = \left(\frac{n\pi c}{L}\right)^2 + \chi_{\text{floor}}^2$$

These are the quantum energy levels of a particle in a box \u2014 emerging from classical wave dynamics on a lattice.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

CHI_0 = 19.0
C = 1.0

def laplacian_1d(f, dx):
    return (np.roll(f, 1) + np.roll(f, -1) - 2 * f) / dx**2

# Create chi-well (quantum box)
N = 400
dx = 1.0
dt = 0.02
x = np.arange(N) * dx
n_steps = 20000

# Chi profile: high walls, low floor (short box for well-separated modes)
chi = np.ones(N) * 5.0       # walls
box_start, box_end = 185, 215
L = (box_end - box_start) * dx  # box width = 30
chi_floor = 0.01  # nearly massless inside box
chi[box_start:box_end] = chi_floor

# Monitor at ~37% of box (avoids nodes for modes 1-5)
monitor_pos = box_start + int(L * 0.37)

# Initialize: Gaussian OFF-CENTER in box (excites both even and odd modes)
init_pos = box_start + L * 0.35  # 35% from left wall
E = np.exp(-(x - init_pos)**2 / (2 * 4**2))
E_prev = E.copy()

def laplacian_1d(f, dx):
    lap = np.zeros_like(f)
    lap[1:-1] = (f[2:] + f[:-2] - 2*f[1:-1]) / dx**2
    return lap

# Collect time history at center of box
history = np.zeros(n_steps)
for step in range(n_steps):
    history[step] = E[monitor_pos]
    E_next = 2*E - E_prev + dt**2 * (C**2 * laplacian_1d(E, dx) - chi**2 * E)
    E_prev, E = E, E_next

# FFT to find frequency peaks
freqs = np.fft.rfftfreq(n_steps, d=dt)
spectrum = np.abs(np.fft.rfft(history))**2

# Simple peak finder
threshold = np.max(spectrum) * 0.005
peaks = []
for i in range(1, len(spectrum) - 1):
    if (spectrum[i] > spectrum[i-1] and spectrum[i] > spectrum[i+1]
            and spectrum[i] > threshold):
        peaks.append(i)

# Theoretical modes: omega_n = sqrt((n*pi*c/L)^2 + chi_floor^2)
print("=" * 60)
print("QUANTIZED ENERGY LEVELS FROM GOV-01")
print("=" * 60)
print(f"\nBox width L = {L}, chi_floor = {chi_floor}")
print(f"\nTheoretical: omega_n = sqrt((n*pi*c/L)^2 + chi_floor^2)")
print(f"\n{'Mode':>6} {'Measured f':>12} {'Theory f':>12} {'Error':>10}")
print("-" * 44)
for idx, n in enumerate(range(1, 8)):
    omega_theory = np.sqrt((n * np.pi * C / L)**2 + chi_floor**2)
    f_theory = omega_theory / (2 * np.pi)
    if idx < len(peaks):
        f_measured = freqs[peaks[idx]]
        error = abs(f_measured - f_theory) / f_theory * 100
        print(f"  n={n:>2}  {f_measured:>12.5f}  {f_theory:>12.5f}  {error:>8.2f}%")
    else:
        print(f"  n={n:>2}  {'(not found)':>12}  {f_theory:>12.5f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Top: chi profile
ax = axes[0]
ax.fill_between(x, chi, alpha=0.3, color='red', label='chi profile')
ax.axhline(chi_floor, color='blue', ls='--', alpha=0.5, label=f'chi_floor = {chi_floor}')
ax.set_ylabel('chi(x)')
ax.set_title('Chi-Well Profile (Particle in a Box)')
ax.legend()
ax.grid(alpha=0.3)

# Bottom: frequency spectrum
ax = axes[1]
ax.semilogy(freqs[:len(spectrum)//4], spectrum[:len(spectrum)//4],
            'b-', linewidth=0.8, label='FFT power spectrum')

# Mark theoretical modes
for n in range(1, 6):
    omega = np.sqrt((n * np.pi * C / L)**2 + chi_floor**2)
    f = omega / (2 * np.pi)
    ax.axvline(f, color='red', ls='--', alpha=0.6, linewidth=1)
    ax.text(f, ax.get_ylim()[1] * 0.5, f' n={n}', color='red',
            fontsize=9, rotation=90, va='top')

ax.set_xlabel('Frequency')
ax.set_ylabel('Power')
ax.set_title('Frequency Spectrum: Peaks Match Quantized Modes')
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('Quantum Energy Levels from GOV-01 in a chi-Well',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Key Result

- Confining a wave in a $\chi$-well produces **discrete frequency modes**
- The modes match the analytical dispersion: $\omega_n = \sqrt{(n\pi c/L)^2 + \chi_{\text{floor}}^2}$
- This is identical to **quantum energy levels** in a particle-in-a-box
- No quantum mechanics was injected \u2014 quantization emerges from wave confinement